# Project MissingNo — training on Kaggle

Runs `training/ppo.py` from the GitHub repository on Kaggle's hardware: the GPU does the learning, the CPU cores run the emulators.

**One-time setup**
1. **Settings → Accelerator → GPU** (T4 ×2 or P100).
2. **Settings → Internet → On** (requires a verified phone number on your Kaggle account).
3. **Add Input → your private dataset** with `pokemon_red.gb` and `red_start.state`. Keep it **private**: the ROM must never be shared.

**Long runs:** use **Save Version → Save & Run All (Commit)**. The notebook runs in the background, even with your PC off, and everything in `/kaggle/working` is kept as the version's output.

**Resuming a run in a new session:** Add Input → *Your Work* → this notebook (the previous version's output), keep the same `RUN_NAME`, and set `RESUME = True`. The notebook finds the last checkpoint by itself. The archive, the curriculum demos and the self-imitation buffer live in memory: after a resume they are rebuilt from scratch, while the model keeps everything it learned.

In [ ]:
# ---------------------------------------------------------------- configuration
REPO = "https://github.com/simoninoProgrammatore/project-MissingNo.git"
BRANCH = "main"

RUN_NAME = "ppo_s1"          # same name across sessions of the same run
SEED = 1
TOTAL_STEPS = 100_000_000    # the whole run, across all sessions
TIME_LIMIT_HOURS = 11.0      # stop cleanly before Kaggle's session limit
RESUME = False               # True: continue RUN_NAME from the previous session's output
NUM_ENVS = None              # None = one game per CPU core
RUN_SPEED_TEST = True        # quick speed diagnosis before training (~1 minute)
EXTRA_ARGS = []              # e.g. ["--ent-coef", "0.02"]

# Example: from scratch to the first badge, stopping there (see docs/SETUP.md):
# RUN_NAME, TOTAL_STEPS = "badge_v22", 200_000_000
# EXTRA_ARGS = ["--reward-version", "v2.2", "--episode-steps", "30000", "--stagnation-steps", "0",
#               "--archive-prob", "0.3", "--curriculum-prob", "0.3", "--sil-coef", "1.0",
#               "--record-every", "8", "--stop-at-goal"]

## 1. Hardware

In [ ]:
import os, subprocess
CPUS = os.cpu_count()
print(f"CPU cores: {CPUS}")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU: enable it in Settings -> Accelerator")

## 2. Code and dependencies

In [ ]:
# The code goes in /tmp so that it does not end up in the notebook's output.
REPO_DIR = "/tmp/missingno"
!rm -rf {REPO_DIR} && git clone -q --depth 1 --branch {BRANCH} {REPO} {REPO_DIR}
!cd {REPO_DIR} && git log -1 --format="Code version: %h %s"

# PyTorch is already installed on Kaggle (with GPU support): we only add what is missing.
!pip install -q pyboy gymnasium tensorboard

# Our packages are used straight from the source folders, nothing to build.
SRC = ":".join(f"{REPO_DIR}/packages/{p}/src" for p in ["core", "games", "envs", "agents"])
os.environ["PYTHONPATH"] = SRC + ":" + os.environ.get("PYTHONPATH", "")
!python --version

## 3. ROM, start state and outputs

In [ ]:
from pathlib import Path
import shutil

def find(name):
    hits = list(Path("/kaggle/input").rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found in /kaggle/input: add your private dataset as input")
    return hits[0]

for name, dest in [("pokemon_red.gb", "roms"), ("red_start.state", "states")]:
    src = find(name)
    shutil.copy(src, f"{REPO_DIR}/{dest}/{name}")
    print(f"{name}: {src}")

# Runs are written to /kaggle/working/runs, which Kaggle keeps as output.
OUT = Path("/kaggle/working/runs")
OUT.mkdir(parents=True, exist_ok=True)
link = Path(REPO_DIR) / "runs"
if not link.exists():
    link.symlink_to(OUT)
print(f"Runs will be saved in {OUT}")

## 4. Resume (only if `RESUME = True`)

In [ ]:
resume_args = []
if RESUME:
    candidates = [p for p in Path("/kaggle/input").rglob("latest.pt") if p.parent.parent.name == RUN_NAME]
    if not candidates:
        raise FileNotFoundError(f"No runs/{RUN_NAME}/checkpoints/latest.pt in the inputs: add the previous output")
    previous = candidates[0].parent.parent
    shutil.copytree(previous, OUT / RUN_NAME, dirs_exist_ok=True)
    resume_args = ["--resume", f"runs/{RUN_NAME}/checkpoints/latest.pt"]
    print(f"Resuming {RUN_NAME} from {previous}")
else:
    print("Starting a new run.")

## 5. Speed test (optional)

In [ ]:
if RUN_SPEED_TEST:
    envs = sorted({2, max(1, CPUS // 2), CPUS})
    !cd {REPO_DIR} && python scripts/diagnose_speed.py --envs {" ".join(map(str, envs))} --seconds 5 --torch-threads 1 2>&1 | grep -v -i "warn\|sdl"

## 6. Training

In [ ]:
import sys
num_envs = NUM_ENVS or CPUS
cmd = [
    sys.executable, "training/ppo.py",
    "--run-name", RUN_NAME,
    "--seed", str(SEED),
    "--total-steps", str(TOTAL_STEPS),
    "--num-envs", str(num_envs),
    "--torch-threads", "1",
    "--device", "auto",
    "--time-limit-hours", str(TIME_LIMIT_HOURS),
    *resume_args,
    *EXTRA_ARGS,
]
print(" ".join(cmd[1:]), "\n")

# Stream the output live; skip the noisy library warnings.
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if "warn" not in line.lower() and "sdl" not in line.lower():
        print(line, end="")
print("Exit code:", proc.wait())

## 7. Results

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

acc = EventAccumulator(str(OUT / RUN_NAME), size_guidance={"scalars": 0})
acc.Reload()
tags = acc.Tags()["scalars"]

def series(tag):
    events = acc.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for tag in sorted(t for t in tags if t.startswith("milestones/") and t.endswith("_rate")):
    axes[0].plot(*series(tag), label=tag.split("/")[1].replace("_rate", ""))
axes[0].set_title("Milestone rate (last 50 episodes)"); axes[0].set_ylim(-0.05, 1.05); axes[0].legend()
for tag, ax in [("episode/tiles_visited", axes[1]), ("episode/return", axes[2])]:
    if tag in tags:
        ax.plot(*series(tag), alpha=0.5); ax.set_title(tag)
for ax in axes:
    ax.set_xlabel("steps")
plt.tight_layout(); plt.show()

if "charts/SPS" in tags:
    print(f"Last SPS: {series('charts/SPS')[1][-1]:,.0f}")

## 8. Goal replay and GIFs

If the run reached the final goal (`--stop-at-goal`), its replay is in `replays/`: download it and watch the exact game at normal speed on your PC with `scripts/replay.py`. The GIFs of the record-breaking episodes are in `gifs/`.


In [ ]:
run_out = OUT / RUN_NAME
for sub in ("replays", "gifs"):
    files = sorted((run_out / sub).glob("*")) if (run_out / sub).exists() else []
    print(f"{sub}: {len(files)} file(s)")
    for f in files[-10:]:
        print("  ", f.name)
winner = run_out / "checkpoints" / "winner.pt"
print("\nFINAL GOAL REACHED: model in checkpoints/winner.pt" if winner.exists() else "\nFinal goal not reached yet: resume in a new session (RESUME = True).")

## 9. Download

Everything in `/kaggle/working/runs/<RUN_NAME>` (curves, checkpoints, config) is in the version's **Output** tab.
Download it, put it in `runs/` on your PC, then:

```
uv run tensorboard --logdir runs
uv run python scripts/watch.py --checkpoint runs/<RUN_NAME>/checkpoints/latest.pt
```